# Task 2 – RAG from scratch (no LangChain, no LlamaIndex)

**Document:** the Llama 2 paper (77 pages, about 42,000 words). **Pipeline:** cut the text into word chunks → embed them with `bge-small-en-v1.5` → cosine search in plain NumPy → give the best chunks to a small LLM (Qwen2.5-1.5B-Instruct) that answers from them only.
**Run:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all* (a few minutes).

| The task asks for | Where (code cells are numbered `# 1` to `# 5`) |
|---|---|
| 15 test questions, each with the source passage | cell 2 |
| Recall@5 for two chunk sizes, which is better and why | cell 3 |
| 2 retrieval failures and what would fix them | cell 4 |
| BM25 and re-ranker in your own words | last cell (drafts to rewrite) |

**Important:** the 15 questions in cell 2 are placeholders so that the notebook runs end to end. The task asks for questions *you* wrote: read the paper, write your own, and copy the passage that answers each one exactly as it appears in the text (at most 25 words). Cell 2 checks every passage.

In [1]:
# 1 - setup: download the paper and extract its text
!pip install -q pymupdf sentence-transformers

import re, unicodedata, requests
import numpy as np, pymupdf, torch
from sentence_transformers import SentenceTransformer

URL = "https://arxiv.org/pdf/2307.09288v2"               # Llama 2 paper, 77 pages. Any public PDF with >= 30 pages works.
SIZES, OVERLAP, K = (100, 300), 25, 5                    # the two chunk sizes (in words), words shared by neighbouring chunks, chunks retrieved
device = "cuda" if torch.cuda.is_available() else "cpu"

r = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=120); r.raise_for_status()
open("doc.pdf", "wb").write(r.content)
pages = [re.sub(r"\s+", " ", unicodedata.normalize("NFKC", p.get_text())).strip() for p in pymupdf.open("doc.pdf")]

words, page_of = [], []                                  # every word of the document and the page it is on
for n, text in enumerate(pages, 1):
    words += text.split(); page_of += [n] * len(text.split())
full = " ".join(words)
assert len(pages) >= 30, "the task needs a document of at least 30 pages"
print(len(pages), "pages,", len(words), "words")

77 pages, 42353 words


## Test questions
A retrieval **hit** means that one of the retrieved chunks contains the source passage verbatim. The passage is kept short (at most `OVERLAP` = 25 words) on purpose: with that rule it always fits completely inside at least one chunk of either size, so a miss is a retrieval problem and not an unlucky chunk boundary.

In [2]:
# 2 - 15 test questions, each with the passage that answers it (verbatim from the paper, at most 25 words).
# PLACEHOLDERS so the notebook runs: replace them with questions you wrote yourself.
QA = [
    ("How many tokens was Llama 2 pretrained on?",
     "We trained on 2 trillion tokens of data"),
    ("Which of the Llama 2 model sizes use grouped-query attention?",
     "Bigger models — 34B and 70B — use Grouped-Query Attention (GQA) for improved inference scalability."),
    ("What learning-rate schedule and warmup did they use for pretraining?",
     "We use a cosine learning rate schedule, with warmup of 2000 steps, and decay final learning rate down to 10% of the peak learning rate."),
    ("How big is the tokenizer's vocabulary?",
     "The total vocabulary size is 32k tokens."),
    ("How much CO2 did pretraining emit in total?",
     "We estimate the total emissions for training to be 539 tCO2eq"),
    ("How many GPU hours did pretraining take, and on which GPUs?",
     "A cumulative of 3.3M GPU hours of computation was performed on hardware of type A100-80GB"),
    ("How many supervised fine-tuning annotations did they end up collecting?",
     "We stopped annotating SFT after collecting a total of 27,540 annotations."),
    ("What batch size and sequence length did they use for supervised fine-tuning?",
     "a weight decay of 0.1, a batch size of 64, and a sequence length of 4096 tokens"),
    ("For how many epochs was the reward model trained?",
     "We train for one epoch over the training data."),
    ("What KL penalty did they use in PPO for the biggest models?",
     "and for the 34B and 70B models, we set β = 0.005."),
    ("What problem is Ghost Attention meant to solve?",
     "RLHF models tended to forget the initial instruction after a few turns of dialogue"),
    ("How does safety context distillation work?",
     "This involves generating safer model responses by prefixing a prompt with a safety preprompt"),
    ("Why was the 34B model not released together with the others?",
     "We are delaying the release of the 34B model due to a lack of time to sufficiently red team."),
    ("How did the authors give the chat model a sense of time?",
     "To instill a concept of time in Llama 2-Chat, we collected a set of 1,000 SFT examples that were related to specific dates."),
    ("What score did Llama 2-Chat get on the ASDiv math benchmark when using tools?",
     "Toolformer 40.4 29.4 44.0 Llama 2-Chat 67.1 69.2 82.4"),
]
for i, (q, g) in enumerate(QA, 1):
    assert len(g.split()) <= OVERLAP and full.count(g) == 1, f"Q{i}: the passage must be at most {OVERLAP} words and occur exactly once in the text"
    page = next((n for n, t in enumerate(pages, 1) if g in t), None)
    assert page, f"Q{i}: the passage must not cross a page break"
    print(f"{i:2d}. [page {page}] {q}\n    source: {g}")

 1. [page 5] How many tokens was Llama 2 pretrained on?
    source: We trained on 2 trillion tokens of data
 2. [page 6] Which of the Llama 2 model sizes use grouped-query attention?
    source: Bigger models — 34B and 70B — use Grouped-Query Attention (GQA) for improved inference scalability.
 3. [page 5] What learning-rate schedule and warmup did they use for pretraining?
    source: We use a cosine learning rate schedule, with warmup of 2000 steps, and decay final learning rate down to 10% of the peak learning rate.
 4. [page 6] How big is the tokenizer's vocabulary?
    source: The total vocabulary size is 32k tokens.
 5. [page 7] How much CO2 did pretraining emit in total?
    source: We estimate the total emissions for training to be 539 tCO2eq
 6. [page 7] How many GPU hours did pretraining take, and on which GPUs?
    source: A cumulative of 3.3M GPU hours of computation was performed on hardware of type A100-80GB
 7. [page 9] How many supervised fine-tuning annotations did the

## Retrieval accuracy for two chunk sizes
**recall@k** = share of the 15 questions for which a hit is among the top-k chunks. Comparing at the same `k` favours big chunks, because 5 chunks of 300 words show the retriever 3× more text than 5 chunks of 100 words. So the last number compares both sizes at the **same amount of text** (about 1,500 words): top-15 small chunks against top-5 big chunks.

In [3]:
# 3 - chunk, embed, search with plain NumPy, measure recall for both chunk sizes
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
embed = lambda texts: embedder.encode(texts, normalize_embeddings=True, batch_size=32)       # unit vectors: dot product = cosine similarity
embed_queries = lambda qs: embed(["Represent this sentence for searching relevant passages: " + q for q in qs])   # bge's query prefix
Q = embed_queries([q for q, _ in QA])

store, ranks = {}, {}
for size in SIZES:
    starts = range(0, len(words), size - OVERLAP)
    chunks = [" ".join(words[i:i + size]) for i in starts]
    E = embed(chunks)
    store[size] = chunks, [page_of[i] for i in starts], E                # text, first page, vectors
    order = np.argsort(-(Q @ E.T), axis=1)                               # shortcut: brute-force NumPy is instant for thousands of chunks; use FAISS beyond ~1M
    ranks[size] = np.array([next(r for r, j in enumerate(order[i], 1) if QA[i][1] in chunks[j]) for i in range(len(QA))])   # rank of the first hit
    budget = 1500 // size
    print(f"{size:>3}-word chunks ({len(chunks):>3} chunks): recall@1 {np.mean(ranks[size] <= 1):.0%} | recall@{K} {np.mean(ranks[size] <= K):.0%}"
          f" | recall@{budget} (about 1,500 words of text) {np.mean(ranks[size] <= budget):.0%}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

100-word chunks (565 chunks): recall@1 33% | recall@5 73% | recall@15 (about 1,500 words of text) 93%
300-word chunks (155 chunks): recall@1 33% | recall@5 80% | recall@5 (about 1,500 words of text) 80%


## Where retrieval fails
For each chunk size, the questions whose hit was ranked worst (rank 1 would be perfect). Pick **two** failures, and for each say why the right chunk was not on top and what would fix it. Ideas are in the last cell.

In [4]:
# 4 - the worst-ranked questions for each chunk size
for size, (chunks, cpages, E) in store.items():
    order = np.argsort(-(Q @ E.T), axis=1)
    print(f"\n===== {size}-word chunks =====")
    for i in [i for i in np.argsort(-ranks[size])[:3] if ranks[size][i] > 1]:
        top = order[i][0]
        print(f"Q{i + 1}: {QA[i][0]}\n  the chunk with the answer was ranked #{ranks[size][i]}; source passage: \"{QA[i][1]}\""
              f"\n  rank-1 chunk instead (page {cpages[top]}): {chunks[top][:250]}...\n")


===== 100-word chunks =====
Q15: What score did Llama 2-Chat get on the ASDiv math benchmark when using tools?
  the chunk with the answer was ranked #106; source passage: "Toolformer 40.4 29.4 44.0 Llama 2-Chat 67.1 69.2 82.4"
  rank-1 chunk instead (page 3): series of helpfulness and safety benchmarks we tested, Llama 2-Chat models generally perform better than existing open-source models. They also appear to be on par with some of the closed-source models, at least on the human evaluations we performed ...

Q9: For how many epochs was the reward model trained?
  the chunk with the answer was ranked #7; source passage: "We train for one epoch over the training data."
  rank-1 chunk instead (page 53): Batch 1 Batch 2 Batch 3 Batch 4 Batch 5 Batch 6 Batch 7 Batch 8 Batch 9 Batch 10 Batch 11 Batch 12 Reward Annotation Stage 0.45 0.50 0.55 0.60 0.65 0.70 0.75 0.80 Reward Score Max wrt 20 samples Med wrt 20 samples Figure 26: Annotation curriculum. Ev...

Q3: What learning-rate schedule 

## The question-answering system
Retrieve the top chunks, put them in the prompt, let a small LLM answer from them only. Any LLM works here; this one runs on the free GPU without an API key.

In [5]:
# 5 - retrieval + LLM answer
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(LLM)
llm = AutoModelForCausalLM.from_pretrained(LLM, dtype=torch.float16 if device == "cuda" else torch.float32).to(device)

def ask(question, size=300, k=K):
    chunks, cpages, E = store[size]
    top = np.argsort(-(embed_queries([question])[0] @ E.T))[:k]
    context = "\n\n".join(f"[page {cpages[j]}] {chunks[j]}" for j in top)
    msgs = [{"role": "system", "content": "Answer the question using only the context. If the answer is not in the context, say you don't know. Be brief."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}]
    x = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True), return_tensors="pt", add_special_tokens=False).to(device)
    out = llm.generate(**x, max_new_tokens=120, do_sample=False, repetition_penalty=1.0)     # 1.0: the default 1.1 would punish copying numbers from the context
    return tok.decode(out[0, x["input_ids"].shape[1]:], skip_special_tokens=True).strip(), sorted({cpages[j] for j in top})

for q in [QA[0][0], QA[5][0], QA[10][0]]:
    answer, cited = ask(q)
    print(f"Q: {q}\nA: {answer}\n   (retrieved from pages {cited})\n")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Q: How many tokens was Llama 2 pretrained on?
A: 2 trillion tokens
   (retrieved from pages [4, 5, 6, 56, 77])

Q: How many GPU hours did pretraining take, and on which GPUs?
A: The pretraining of the Llama 2 models took a cumulative 3.3M GPU hours of computation on hardware of type A100-80GB (TDP of 350-400W).
   (retrieved from pages [5, 6, 7, 47, 77])

Q: What problem is Ghost Attention meant to solve?
A: Ghost Attention is meant to solve the problem of multi-turn memory issues in dialogue systems.
   (retrieved from pages [8, 15, 16, 26, 56])



## Drafts for the written parts (rewrite them in your own words, and check them against your own results)

**Which chunk size worked better, and why.** Read your numbers from cell 3. Typical reasons: *small chunks* (about 100 words) give a sharper embedding, because one chunk covers one idea, so the right chunk ranks high, but a fact can be split across a boundary, and the chunk may lack the context needed to answer. *Big chunks* (about 300 words) keep the context together and so catch more answers within the top 5, but the embedding becomes an average over several topics (diluted), the LLM receives more text per hit, and top-5 simply covers 3× more of the document. The equal-text number in cell 3 separates "big chunks are better" from "big chunks show more text". With only 15 questions, one question moves recall by 6.7 points, so a gap of one or two questions is within noise: if that is what you see, say the sizes are about equal, and use the ranks and the equal-text number as extra evidence.

**What BM25 adds to embedding search.** BM25 is classic keyword scoring: a passage scores higher when it contains the query's words, rare words count more (inverse document frequency), and repeating a word or writing a very long passage helps only up to a limit. Embeddings capture meaning, so they cope with paraphrases but blur exact strings such as names, numbers, acronyms and identifiers (`GAtt`, `34B`, `β = 0.005`). BM25 catches exactly those, needs no model, and fails where words differ but meaning matches. Hybrid search runs both and merges the two ranked lists (for example with reciprocal rank fusion).

**What a re-ranker does.** The embedder encodes the question and each chunk separately (a bi-encoder), which is fast because chunk vectors are computed once, but coarse. A re-ranker (a cross-encoder) reads the question and one candidate chunk *together* and outputs a relevance score, so it can check whether the passage really answers the question (numbers, negation, which stage of training). It is too slow to run on the whole corpus, so it re-orders only the top 20 to 50 candidates from the fast search, and the best 3 to 5 go to the LLM.

**What would fix a failure** (match the fix to your failure): exact numbers, names or table rows → BM25 or hybrid search, or table-aware parsing that keeps a table row with its caption; several similar passages for different stages (for example batch sizes) → a re-ranker, or put the section title into each chunk; an answer cut by a chunk boundary → more overlap, sentence- or section-aware chunking; a question phrased very differently from the text → query rewriting or a stronger embedding model.